# Notebook 03 — Capa Gold, Indicadores, Visualizaciones en Databricks y Consultas con Genie IA
**Proyecto:** Implementación de Soluciones BI, Big Data y Experimentación — Unity Perú SAC
**Arquitectura:** Medallion Lakehouse (Silver $\rightarrow$ Gold + Dashboard Lakeview + Databricks Genie)

### Tablas Analíticas Persistidas en Capa Gold (`unity_gold`)
1. `unity_gold.kpi_mensual`: Entradas, salidas, rotación, TPP, inversión retenida, stock acumulado (Window) y cumplimiento del plan (VM).
2. `unity_gold.kpi_categoria`: Exposición financiera, rotación, TPP y cumplimiento por categoría tecnológica.
3. `unity_gold.stock_componentes`: Disponibilidad neta vs. stock mínimo y exactitud de conteos cíclicos por componente.
4. `unity_gold.consumo_proyectos`: Unidades y presupuesto despachado por proyecto de misión crítica.
5. `unity_gold.telemetria_almacen`: Condiciones térmicas y de humedad agregadas por ubicación y mes.


In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("UnityPeru_03_Gold_Analytics").getOrCreate()
spark.sql("CREATE SCHEMA IF NOT EXISTS unity_gold")
FORMATO_TABLA = "delta"

df_mov = spark.table("unity_silver.movimientos")
df_plan = spark.table("unity_silver.plan_demanda")
df_con = spark.table("unity_silver.conteos_ciclicos")
df_sen = spark.table("unity_silver.sensores_iot")


In [ ]:
# 1. Tabla Gold KPI Mensual con Funciones de Ventana (Window)
plan_mes = df_plan.groupBy("anio", "mes").agg(
    F.sum("entradas_plan").alias("entradas_plan"),
    F.sum("salidas_plan").alias("salidas_plan")
)

mov_mes = df_mov.groupBy("anio", "mes").agg(
    F.sum(F.when(F.col("es_entrada") == 1, F.col("cantidad")).otherwise(0)).alias("unidades_entradas"),
    F.sum(F.when(F.col("es_salida_real") == 1, F.col("cantidad")).otherwise(0)).alias("unidades_salidas"),
    F.round(F.sum(F.when(F.col("es_entrada") == 1, F.col("costo_total")).otherwise(0)), 2).alias("costo_entradas"),
    F.round(F.sum(F.when(F.col("es_salida_real") == 1, F.col("costo_total")).otherwise(0)), 2).alias("costo_salidas"),
    F.round(F.avg("dias_permanencia"), 2).alias("tpp_dias")
)

win_acum = Window.orderBy("anio", "mes").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_gold_mensual = (
    mov_mes.join(plan_mes, on=["anio", "mes"], how="left")
    .withColumn("rotacion_flujo", F.round(F.col("unidades_salidas") / F.col("unidades_entradas"), 4))
    .withColumn("balance_neto_mes", F.col("unidades_entradas") - F.col("unidades_salidas"))
    .withColumn("stock_acumulado", F.sum("balance_neto_mes").over(win_acum))
    .withColumn("vm_entradas_pct", F.round((F.col("unidades_entradas") - F.col("entradas_plan")) / F.col("entradas_plan") * 100, 2))
    .withColumn("vm_salidas_pct", F.round((F.col("unidades_salidas") - F.col("salidas_plan")) / F.col("salidas_plan") * 100, 2))
    .orderBy("anio", "mes")
)

df_gold_mensual.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_gold.kpi_mensual")
display(df_gold_mensual) if "display" in globals() else df_gold_mensual.show()


In [ ]:
# 2. Tablas Gold por Categoría, Componente y Proyecto
df_gold_categoria = (
    df_mov.groupBy("categoria")
    .agg(
        F.countDistinct("codigo_componente").alias("componentes"),
        F.sum(F.when(F.col("es_entrada") == 1, F.col("cantidad")).otherwise(0)).alias("unidades_entradas"),
        F.sum(F.when(F.col("es_salida_real") == 1, F.col("cantidad")).otherwise(0)).alias("unidades_salidas"),
        F.round(F.sum(F.when(F.col("es_entrada") == 1, F.col("costo_total")).otherwise(0)), 2).alias("costo_entradas"),
        F.round(F.sum(F.when(F.col("es_salida_real") == 1, F.col("costo_total")).otherwise(0)), 2).alias("costo_salidas"),
        F.round(F.avg("dias_permanencia"), 2).alias("tpp_dias")
    )
    .withColumn("rotacion_flujo", F.round(F.col("unidades_salidas") / F.col("unidades_entradas"), 4))
    .withColumn("inversion_retenida", F.round(F.col("costo_entradas") - F.col("costo_salidas"), 2))
    .orderBy(F.desc("costo_entradas"))
)

df_gold_proyectos = (
    df_mov.filter(F.col("es_salida_real") == 1)
    .groupBy("codigo_proyecto", "nombre_proyecto")
    .agg(
        F.count("*").alias("num_despachos"),
        F.sum("cantidad").alias("unidades_consumidas"),
        F.round(F.sum("costo_total"), 2).alias("costo_consumido"),
        F.round(F.avg("dias_permanencia"), 2).alias("tpp_dias")
    )
    .orderBy(F.desc("unidades_consumidas"))
)

df_gold_categoria.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_gold.kpi_categoria")
df_gold_proyectos.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_gold.consumo_proyectos")

df_gold_categoria.show(truncate=False)
df_gold_proyectos.show(truncate=False)


In [ ]:
# 2.3 Disponibilidad por componente y telemetría agregada
exactitud = df_con.groupBy("codigo_componente").agg(F.count("*").alias("num_conteos"), F.round(F.avg("exactitud_linea") * 100, 2).alias("exactitud_pct"))
componentes = (df_mov.groupBy("codigo_componente", "categoria", "criticidad", "stock_minimo", "precio_unitario")
    .agg(F.sum(F.when(F.col("es_entrada") == 1, F.col("cantidad")).otherwise(0)).alias("entradas"),
         F.sum(F.when(F.col("es_salida_real") == 1, F.col("cantidad")).otherwise(0)).alias("salidas"),
         F.round(F.avg("dias_permanencia"), 2).alias("tpp_dias"))
    .withColumn("stock_neto", F.col("entradas") - F.col("salidas"))
    .withColumn("cumple_stock_minimo", F.when(F.col("stock_neto") >= F.col("stock_minimo"), 1).otherwise(0))
    .join(exactitud, "codigo_componente", "left"))
telemetria = (df_sen.groupBy("codigo_ubicacion", "mes")
    .agg(F.count("*").alias("lecturas_totales"), F.round(F.avg("temperatura_c"), 2).alias("temp_promedio_c"),
         F.round(F.avg("humedad_pct"), 2).alias("hum_promedio_pct")))
componentes.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_gold.stock_componentes")
telemetria.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_gold.telemetria_almacen")
print("componentes:", componentes.count(), "telemetría:", telemetria.count())


## 3. Conciliación Cruzada BI vs. Big Data y Consultas Analíticas para Databricks Genie (IA)
Propuesta pendiente de despliegue: conectar `unity_gold.*` a un **Espacio Genie de Databricks** y verificar sus respuestas contra SQL y el respaldo BI:
- **Prompt Genie 1:** *"¿Cuál es la categoría tecnológica con mayor inversión retenida y qué porcentaje del costo total de entradas representa?"*
- **Prompt Genie 2:** *"Muestra la evolución mensual del TPP en días frente a las unidades de entrada y salida de enero a setiembre de 2026."*
- **Prompt Genie 3:** *"¿Qué proyecto ha consumido más unidades y cuál es su tiempo promedio de despacho?"*


In [ ]:
# Verificación SQL exacta de los KPI oficiales en Databricks
spark.sql('''
    SELECT
        COUNT(*) AS movimientos_verificados,
        SUM(CASE WHEN es_entrada = 1 THEN cantidad ELSE 0 END) AS entradas_u,
        SUM(CASE WHEN es_salida_real = 1 THEN cantidad ELSE 0 END) AS salidas_u,
        ROUND(SUM(CASE WHEN es_entrada = 1 THEN costo_total ELSE 0 END) -
              SUM(CASE WHEN es_salida_real = 1 THEN costo_total ELSE 0 END), 2) AS inversion_retenida_soles,
        ROUND(AVG(dias_permanencia), 6) AS tpp_exacto_dias,
        ROUND(SUM(CASE WHEN es_salida_real = 1 THEN cantidad ELSE 0 END) * 1.0 /
              SUM(CASE WHEN es_entrada = 1 THEN cantidad ELSE 0 END), 4) AS rotacion_global
    FROM unity_silver.movimientos
''').show()
